# Predictive Analytics Project

Luke M. Hebert

In [10]:
# Loaded all the libraries because I'll be using a whole lot

# Core Libraries
import warnings
import os
import glob
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from scipy.spatial.distance import cosine

warnings.filterwarnings("ignore")

# Clean plot style
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Optional display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Scikit-Learn: Model Selection / Validation
from sklearn.model_selection import (
    train_test_split,
    KFold,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
)

# Scikit-Learn: Preprocessing
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

# Scikit-Learn: Regression Models
from sklearn.linear_model import LinearRegression

# Scikit-Learn: Classification Models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

# Scikit-Learn: Ensemble Models
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier,
    BaggingClassifier,
)

# Scikit-Learn: Metrics
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

from sklearn.metrics.pairwise import cosine_similarity

# Gradient Boosting Libraries
import xgboost as xgb
import lightgbm as lgb
from xgboost import XGBRegressor as xgb

# Interpretability
import shap

# Association Rules
from mlxtend.frequent_patterns import apriori, association_rules


print("✅ Libraries loaded!")

✅ Libraries loaded!


In [11]:
# Data repair for raw Russell 3000 constituents file
# Found list here: https://www.ishares.com/us/products/239714/ishares-russell-3000-etf

# Imports
from pathlib import Path
import pandas as pd

# Use current working directory in Jupyter
script_dir = Path.cwd()

# Build paths for the raw input file and cleaned output file
input_path = script_dir / "constituents_raw.csv"
output_path = script_dir / "constituents.csv"

# Read in just the Symbol column from the raw file
russell3000 = pd.read_csv(input_path)[["Symbol"]].copy()

# Rename Symbol to ticker
russell3000 = russell3000.rename(columns={"Symbol": "ticker"})

# Clean up ticker values just in case:
# turn everything into strings, strip spaces, and make uppercase
russell3000["ticker"] = russell3000["ticker"].astype(str).str.strip().str.upper()

# Fix the few tickers that Yahoo Finance writes differently
russell3000["ticker"] = russell3000["ticker"].replace({
    "BRK.B": "BRK-B",
    "BF.B": "BF-B"
})

# Quick check
print(russell3000.shape)
print(russell3000.head())

# Save the cleaned file
russell3000.to_csv(output_path, index=False)

(2586, 1)
  ticker
0   NVDA
1   AAPL
2   MSFT
3   AMZN
4  GOOGL


In [12]:
import time
import requests
import numpy as np
import pandas as pd

# =========================================================
# CONFIG
# =========================================================
API_KEY = "252eb4bc1b6d678c260e3fdb340ff24d9e298a40b638d860"
BASE_URL = "https://www.stockainsights.com/api-si/v1"

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

CONSTITUENTS_CSV = "constituents.csv"
RAW_OUTPUT_CSV = "russell3000_quarterly_raw_panel.csv"
LABELED_OUTPUT_CSV = "russell3000_quarterly_buyback_labeled.csv"
API_STATUS_OUTPUT_CSV = "api_build_status.csv"

START_YEAR = 2014
END_YEAR = 2025

REQUEST_SLEEP_SECONDS = 0.15
MAX_RETRIES = 5
REQUEST_TIMEOUT = 45

# Hybrid label threshold
# Assumes shares_change_percentage is in percent units,
# e.g. -0.8 means -0.8%
BUYBACK_THRESHOLD_PCT = -1.25

# For API compatibility on dotted tickers
TICKER_API_OVERRIDES = {
    "BRK-B": "BRK.B",
    "BF-B": "BF.B",
}

# =========================================================
# HTTP HELPERS
# =========================================================
session = requests.Session()
session.headers.update(HEADERS)


def safe_get(endpoint, params=None, retries=MAX_RETRIES):
    """
    GET with retry handling.
    Returns parsed JSON or None.
    """
    url = f"{BASE_URL}{endpoint}"

    for attempt in range(retries):
        try:
            response = session.get(url, params=params, timeout=REQUEST_TIMEOUT)

            if response.status_code == 200:
                time.sleep(REQUEST_SLEEP_SECONDS)
                return response.json()

            elif response.status_code == 429:
                wait = 2.0 + attempt
                print(f"[429] {endpoint} | sleeping {wait:.1f}s")
                time.sleep(wait)

            elif response.status_code in (400, 401, 403, 404):
                try:
                    err = response.json()
                except Exception:
                    err = response.text

                print(f"[ERROR] {endpoint} | status={response.status_code} | {err}")
                return None

            else:
                try:
                    err = response.json()
                except Exception:
                    err = response.text

                wait = 1.0 + attempt
                print(f"[ERROR] {endpoint} | status={response.status_code} | {err}")
                print(f"[RETRY] {endpoint} | sleeping {wait:.1f}s")
                time.sleep(wait)

        except Exception as e:
            wait = 1.0 + attempt
            print(f"[EXCEPTION] {endpoint} | {e}")
            print(f"[RETRY] {endpoint} | sleeping {wait:.1f}s")
            time.sleep(wait)

    return None


def extract_data(payload):
    if payload is None:
        return None
    return payload.get("data")


# =========================================================
# DATAFRAME UTILITIES
# =========================================================
def load_constituents(csv_path):
    """
    Reads current Russell 3000 constituent list from local CSV.
    Required column: ticker
    """
    df = pd.read_csv(csv_path)

    possible_cols = [c for c in df.columns if c.lower().strip() == "ticker"]
    if not possible_cols:
        raise ValueError("constituents.csv must contain a 'ticker' column.")

    tickers = (
        df[possible_cols[0]]
        .astype(str)
        .str.strip()
        .str.upper()
        .replace("", np.nan)
        .dropna()
        .drop_duplicates()
        .tolist()
    )

    print(f"Loaded {len(tickers)} tickers from {csv_path}")
    return tickers


def get_api_ticker_candidates(raw_ticker):
    """
    Some APIs prefer dots for certain classes, while your CSV may use hyphens.
    Try the override first, then the raw ticker.
    """
    if raw_ticker in TICKER_API_OVERRIDES:
        return [TICKER_API_OVERRIDES[raw_ticker], raw_ticker]
    return [raw_ticker]


def standardize_period_df(df, raw_ticker=None, api_ticker_used=None):
    """
    Standardize shared period columns and keep only quarterly rows.
    """
    if df is None or df.empty:
        return pd.DataFrame()

    out = df.copy()

    if raw_ticker is not None:
        out["ticker"] = raw_ticker

    if api_ticker_used is not None:
        out["api_ticker_used"] = api_ticker_used

    if "period_end_date" in out.columns:
        out["period_end_date"] = pd.to_datetime(out["period_end_date"], errors="coerce")

    if "fiscal_year" in out.columns:
        out["fiscal_year"] = pd.to_numeric(out["fiscal_year"], errors="coerce")

    if "fiscal_period" in out.columns:
        out["fiscal_period"] = out["fiscal_period"].astype(str).str.upper().str.strip()

    if "fiscal_period" in out.columns:
        out = out[out["fiscal_period"].isin(["Q1", "Q2", "Q3", "Q4"])].copy()

    if "fiscal_year" in out.columns:
        out = out[(out["fiscal_year"] >= START_YEAR) & (out["fiscal_year"] <= END_YEAR)].copy()

    return out.reset_index(drop=True)


def prefix_non_keys(df, prefix):
    """
    Prefix non-key columns to avoid collisions.
    """
    if df is None or df.empty:
        return pd.DataFrame()

    keys = {
        "ticker",
        "api_ticker_used",
        "fiscal_year",
        "fiscal_period",
        "period_end_date",
    }

    rename_map = {c: f"{prefix}{c}" for c in df.columns if c not in keys}
    return df.rename(columns=rename_map)


def dedupe_panel_keys(df):
    """
    Force one row per (ticker, fiscal_year, fiscal_period).
    Keep the row with the most non-null values.
    """
    if df is None or df.empty:
        return pd.DataFrame()

    out = df.copy()
    out["_non_null_count"] = out.notna().sum(axis=1)

    sort_cols = ["ticker", "fiscal_year", "fiscal_period", "_non_null_count"]
    ascending = [True, True, True, False]

    if "period_end_date" in out.columns:
        sort_cols.append("period_end_date")
        ascending.append(False)

    out = out.sort_values(sort_cols, ascending=ascending)

    out = out.drop_duplicates(
        subset=["ticker", "fiscal_year", "fiscal_period"],
        keep="first",
    )

    out = out.drop(columns=["_non_null_count"], errors="ignore")
    return out.reset_index(drop=True)


def merge_panel(left, right):
    """
    Merge on (ticker, fiscal_year, fiscal_period) after deduping both sides.
    Coalesce duplicated period_end_date and api_ticker_used.
    """
    if left is None or left.empty:
        return dedupe_panel_keys(right.copy()) if right is not None else pd.DataFrame()

    if right is None or right.empty:
        return dedupe_panel_keys(left.copy())

    left = dedupe_panel_keys(left)
    right = dedupe_panel_keys(right)

    merged = left.merge(
        right,
        on=["ticker", "fiscal_year", "fiscal_period"],
        how="outer",
        suffixes=("", "__dup"),
    )

    # Coalesce period_end_date
    if "period_end_date__dup" in merged.columns:
        if "period_end_date" in merged.columns:
            merged["period_end_date"] = pd.to_datetime(
                merged["period_end_date"], errors="coerce"
            ).combine_first(
                pd.to_datetime(merged["period_end_date__dup"], errors="coerce")
            )
        else:
            merged["period_end_date"] = pd.to_datetime(
                merged["period_end_date__dup"], errors="coerce"
            )

        merged = merged.drop(columns=["period_end_date__dup"])

    # Coalesce api_ticker_used
    if "api_ticker_used__dup" in merged.columns:
        if "api_ticker_used" in merged.columns:
            merged["api_ticker_used"] = merged["api_ticker_used"].combine_first(
                merged["api_ticker_used__dup"]
            )
        else:
            merged["api_ticker_used"] = merged["api_ticker_used__dup"]

        merged = merged.drop(columns=["api_ticker_used__dup"])

    return dedupe_panel_keys(merged)


def quarter_sort_key(series):
    mapping = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}
    return series.map(mapping)


def sort_panel(df):
    """
    Sort panel chronologically.
    """
    out = df.copy()

    if "period_end_date" in out.columns:
        out["period_end_date"] = pd.to_datetime(out["period_end_date"], errors="coerce")
        out = out.sort_values(["ticker", "period_end_date"]).reset_index(drop=True)
    else:
        out["_qsort"] = quarter_sort_key(out["fiscal_period"])
        out = (
            out.sort_values(["ticker", "fiscal_year", "_qsort"])
            .drop(columns="_qsort")
            .reset_index(drop=True)
        )

    return out


def coerce_bool_series(s):
    """
    Robust boolean coercion for values like True/False, 1/0, 'true'/'false'.
    """
    if s is None:
        return pd.Series(dtype=bool)

    s2 = s.copy()

    if s2.dtype == bool:
        return s2.fillna(False)

    s2 = s2.astype(str).str.strip().str.lower()
    true_vals = {"true", "1", "yes", "y", "t"}
    return s2.isin(true_vals)


# =========================================================
# ENDPOINT HELPERS
# =========================================================
def get_endpoint_with_fallback(raw_ticker, endpoint_template, params=None, expect="list"):
    """
    Try API ticker candidates in order until one returns usable data.

    expect: 'list' or 'dict'
    Returns (data, api_ticker_used)
    """
    for api_ticker in get_api_ticker_candidates(raw_ticker):
        payload = safe_get(endpoint_template.format(ticker=api_ticker), params=params)
        data = extract_data(payload)

        if expect == "list" and isinstance(data, list) and len(data) > 0:
            return data, api_ticker

        if expect == "dict" and isinstance(data, dict) and len(data) > 0:
            return data, api_ticker

    return None, None


def get_profile(raw_ticker):
    data, api_ticker_used = get_endpoint_with_fallback(
        raw_ticker,
        "/profile/{ticker}",
        params=None,
        expect="dict",
    )

    if data is None:
        return {}, None

    return data, api_ticker_used


def get_income_statements(raw_ticker):
    data, api_ticker_used = get_endpoint_with_fallback(
        raw_ticker,
        "/income-statements/{ticker}",
        params={"period": "Q", "limit": 100},
        expect="list",
    )

    df = pd.DataFrame(data) if data is not None else pd.DataFrame()
    df = standardize_period_df(df, raw_ticker=raw_ticker, api_ticker_used=api_ticker_used)

    return df, api_ticker_used


def get_balance_sheets(raw_ticker):
    data, api_ticker_used = get_endpoint_with_fallback(
        raw_ticker,
        "/balance-sheets/{ticker}",
        params={"period": "Q", "limit": 100},
        expect="list",
    )

    df = pd.DataFrame(data) if data is not None else pd.DataFrame()
    df = standardize_period_df(df, raw_ticker=raw_ticker, api_ticker_used=api_ticker_used)

    return df, api_ticker_used


def get_cash_flows(raw_ticker):
    data, api_ticker_used = get_endpoint_with_fallback(
        raw_ticker,
        "/cash-flows/{ticker}",
        params={"period": "Q", "limit": 100},
        expect="list",
    )

    df = pd.DataFrame(data) if data is not None else pd.DataFrame()
    df = standardize_period_df(df, raw_ticker=raw_ticker, api_ticker_used=api_ticker_used)

    return df, api_ticker_used


def get_shares_outstanding(raw_ticker):
    data, api_ticker_used = get_endpoint_with_fallback(
        raw_ticker,
        "/shares-outstanding/{ticker}",
        params={"period": "Q", "limit": 100},
        expect="list",
    )

    df = pd.DataFrame(data) if data is not None else pd.DataFrame()
    df = standardize_period_df(df, raw_ticker=raw_ticker, api_ticker_used=api_ticker_used)

    return df, api_ticker_used


# =========================================================
# BUILD ONE COMPANY PANEL
# =========================================================
def build_company_panel(raw_ticker):
    print(f"Building panel for {raw_ticker}...")

    profile, profile_api_ticker = get_profile(raw_ticker)

    income_raw, income_api_ticker = get_income_statements(raw_ticker)
    balance_raw, balance_api_ticker = get_balance_sheets(raw_ticker)
    cash_raw, cash_api_ticker = get_cash_flows(raw_ticker)
    shares_raw, shares_api_ticker = get_shares_outstanding(raw_ticker)

    income = dedupe_panel_keys(prefix_non_keys(income_raw, "inc_"))
    balance = dedupe_panel_keys(prefix_non_keys(balance_raw, "bal_"))
    cash = dedupe_panel_keys(prefix_non_keys(cash_raw, "cf_"))
    shares = dedupe_panel_keys(prefix_non_keys(shares_raw, "shr_"))

    status = {
        "ticker": raw_ticker,
        "profile_api_ticker": profile_api_ticker,
        "income_api_ticker": income_api_ticker,
        "balance_api_ticker": balance_api_ticker,
        "cash_api_ticker": cash_api_ticker,
        "shares_api_ticker": shares_api_ticker,
        "income_rows": len(income),
        "balance_rows": len(balance),
        "cash_rows": len(cash),
        "shares_rows": len(shares),
        "panel_rows": 0,
        "panel_built": 0,
    }

    panel = pd.DataFrame()

    for df in [income, balance, cash, shares]:
        panel = merge_panel(panel, df)

    if panel.empty:
        return pd.DataFrame(), status

    panel["profile_api_ticker_used"] = profile_api_ticker
    panel["sector"] = profile.get("sector")
    panel["industry"] = profile.get("industry")
    panel["sic_code"] = profile.get("sic_code")
    panel["company_name"] = profile.get("name") or profile.get("company_name_full")
    panel["fiscal_year_end"] = profile.get("fiscal_year_end")
    panel["website_url"] = profile.get("website_url")

    exchanges = profile.get("exchanges")
    panel["exchange"] = exchanges[0] if isinstance(exchanges, list) and len(exchanges) > 0 else np.nan

    panel = dedupe_panel_keys(panel)
    panel["fiscal_year"] = pd.to_numeric(panel["fiscal_year"], errors="coerce")
    panel = panel[(panel["fiscal_year"] >= START_YEAR) & (panel["fiscal_year"] <= END_YEAR)].copy()

    panel = sort_panel(panel)

    panel = panel.replace([np.inf, -np.inf], np.nan).infer_objects(copy=False)

    status["panel_rows"] = len(panel)
    status["panel_built"] = 1 if len(panel) > 0 else 0

    return panel, status


# =========================================================
# BUYBACK LABELING
# =========================================================
def add_buyback_labels(raw_df):
    """
    Adds current-quarter and next-quarter buyback labels.

    buyback_quarter = 1 if:
    shr_is_buyback == True OR
    shr_shares_change_percentage <= BUYBACK_THRESHOLD_PCT
    """
    df = raw_df.copy()

    df = dedupe_panel_keys(df)
    df = sort_panel(df)

    if "shr_is_buyback" in df.columns:
        buyback_flag = coerce_bool_series(df["shr_is_buyback"])
    else:
        buyback_flag = pd.Series(False, index=df.index)

    if "shr_shares_change_percentage" in df.columns:
        share_change = pd.to_numeric(df["shr_shares_change_percentage"], errors="coerce")
        share_change_buyback = share_change <= BUYBACK_THRESHOLD_PCT

        df["shr_shares_change_percentage"] = share_change
    else:
        share_change = pd.Series(np.nan, index=df.index)
        share_change_buyback = pd.Series(False, index=df.index)

    df["buyback_quarter_api_flag"] = buyback_flag.astype(int)
    df["buyback_quarter_share_reduction_flag"] = share_change_buyback.astype(int)
    df["buyback_quarter"] = (buyback_flag | share_change_buyback).astype(int)

    # Next-quarter target
    df["buyback_next_quarter"] = df.groupby("ticker")["buyback_quarter"].shift(-1)

    df = df.replace([np.inf, -np.inf], np.nan).infer_objects(copy=False)

    return df


# =========================================================
# MAIN
# =========================================================
def main():
    tickers = load_constituents(CONSTITUENTS_CSV)

    all_panels = []
    all_status = []

    for i, ticker in enumerate(tickers, start=1):
        print(f"\n[{i}/{len(tickers)}] {ticker}")

        try:
            company_panel, status = build_company_panel(ticker)
            all_status.append(status)

            if not company_panel.empty:
                all_panels.append(company_panel)
            else:
                print(f"[WARNING] Empty panel for {ticker}")

        except Exception as e:
            print(f"[FAILED] {ticker} | {e}")

            all_status.append({
                "ticker": ticker,
                "profile_api_ticker": None,
                "income_api_ticker": None,
                "balance_api_ticker": None,
                "cash_api_ticker": None,
                "shares_api_ticker": None,
                "income_rows": 0,
                "balance_rows": 0,
                "cash_rows": 0,
                "shares_rows": 0,
                "panel_rows": 0,
                "panel_built": 0,
                "error": str(e),
            })

    if len(all_panels) == 0:
        raise RuntimeError("No company panels were built.")

    raw_df = pd.concat(all_panels, ignore_index=True)

    # Final global dedupe and sort
    raw_df = dedupe_panel_keys(raw_df)
    raw_df = sort_panel(raw_df)

    # Add buyback labels after the full raw panel is built
    raw_df = add_buyback_labels(raw_df)

    # Save raw panel
    raw_df.to_csv(RAW_OUTPUT_CSV, index=False)

    # Save API status report
    status_df = pd.DataFrame(all_status)
    status_df.to_csv(API_STATUS_OUTPUT_CSV, index=False)

    # Save supervised / labeled panel
    labeled_df = raw_df.dropna(subset=["buyback_next_quarter"]).copy()
    labeled_df["buyback_next_quarter"] = labeled_df["buyback_next_quarter"].astype(int)
    labeled_df.to_csv(LABELED_OUTPUT_CSV, index=False)

    # -----------------------------------------------------
    # SANITY CHECKS
    # -----------------------------------------------------
    raw_dups = raw_df.duplicated(subset=["ticker", "fiscal_year", "fiscal_period"]).sum()
    lab_dups = labeled_df.duplicated(subset=["ticker", "fiscal_year", "fiscal_period"]).sum()

    built_tickers = status_df["panel_built"].sum()
    empty_tickers = (status_df["panel_built"] == 0).sum()

    print("\n==============================")
    print("BUILD COMPLETE")
    print("==============================")
    print(f"Threshold used: {BUYBACK_THRESHOLD_PCT}")
    print(f"Total tickers attempted: {len(tickers)}")
    print(f"Tickers with panels built: {built_tickers}")
    print(f"Tickers with empty/failed panels: {empty_tickers}")
    print(f"Raw panel shape: {raw_df.shape}")
    print(f"Labeled panel shape: {labeled_df.shape}")
    print(f"Raw unique tickers: {raw_df['ticker'].nunique()}")
    print(f"Labeled unique tickers: {labeled_df['ticker'].nunique()}")
    print(f"Raw duplicate key rows: {raw_dups}")
    print(f"Labeled duplicate rows: {lab_dups}")
    print(f"Raw output file: {RAW_OUTPUT_CSV}")
    print(f"Labeled output file: {LABELED_OUTPUT_CSV}")
    print(f"API status output file: {API_STATUS_OUTPUT_CSV}")

    print("\nEndpoint row summary:")
    print(status_df[["income_rows", "balance_rows", "cash_rows", "shares_rows", "panel_rows"]].describe())

    print("\nBuyback quarter distribution:")
    print(raw_df["buyback_quarter"].value_counts(dropna=False).sort_index())

    print("\nBuyback next quarter distribution:")
    print(labeled_df["buyback_next_quarter"].value_counts(dropna=False).sort_index())

    print("\nPositive class rate:")
    print(f"{labeled_df['buyback_next_quarter'].mean():.2%}")

    print("\nRows by fiscal year:")
    print(labeled_df["fiscal_year"].value_counts().sort_index())

    print("\nQuarters per ticker:")
    print(labeled_df.groupby("ticker").size().describe())

    print("\nFirst 30 columns:")
    print(raw_df.columns[:30].tolist())

    print("\nHead:")
    print(labeled_df.head())


if __name__ == "__main__":
    main()

Loaded 2585 tickers from constituents.csv

[1/2585] NVDA
Building panel for NVDA...

[2/2585] AAPL
Building panel for AAPL...

[3/2585] MSFT
Building panel for MSFT...

[4/2585] AMZN
Building panel for AMZN...

[5/2585] GOOGL
Building panel for GOOGL...

[6/2585] AVGO
Building panel for AVGO...

[7/2585] GOOG
Building panel for GOOG...

[8/2585] META
Building panel for META...

[9/2585] TSLA
Building panel for TSLA...

[10/2585] BRKB
Building panel for BRKB...
[ERROR] /profile/BRKB | status=404 | {'error': 'Not found', 'message': 'Company not found for: BRKB', 'status': 404, 'timestamp': '2026-05-30T00:44:55.231Z'}
[WARNING] Empty panel for BRKB

[11/2585] JPM
Building panel for JPM...

[12/2585] LLY
Building panel for LLY...

[13/2585] XOM
Building panel for XOM...

[14/2585] JNJ
Building panel for JNJ...

[15/2585] WMT
Building panel for WMT...

[16/2585] V
Building panel for V...

[17/2585] MU
Building panel for MU...

[18/2585] AMD
Building panel for AMD...

[19/2585] COST
Building

In [13]:
import pandas as pd

raw = pd.read_csv("russell3000_quarterly_raw_panel.csv")
lab = pd.read_csv("russell3000_quarterly_buyback_labeled.csv")

print(raw.shape)
print(lab.shape)
print(raw.duplicated(subset=["ticker", "fiscal_year", "fiscal_period"]).sum())
print(lab.duplicated(subset=["ticker", "fiscal_year", "fiscal_period"]).sum())
print(lab["buyback_next_quarter"].value_counts(normalize=True))
print(lab.columns.tolist()[:60])

(18274, 403)
(17319, 403)
0
0
buyback_next_quarter
0    0.573821
1    0.426179
Name: proportion, dtype: float64
['fiscal_year', 'fiscal_period', 'period_end_date', 'inc_currency', 'inc_revenue', 'inc_revenue_product', 'inc_revenue_services', 'inc_cost_of_revenue', 'inc_gross_profit', 'inc_operating_expenses', 'inc_research_development', 'inc_selling_general_administrative', 'inc_selling_marketing_expenses', 'inc_general_admin_expenses', 'inc_depreciation_amortization', 'inc_depreciation', 'inc_amortization_of_intangibles', 'inc_stock_based_compensation', 'inc_operating_income', 'inc_interest_expense', 'inc_interest_income', 'inc_other_income_expense', 'inc_income_before_taxes', 'inc_income_tax_expense', 'inc_net_income_continuing_operations', 'inc_net_income_discontinued_operations', 'inc_net_income', 'inc_basic_shares_outstanding', 'inc_diluted_shares_outstanding', 'inc_weighted_average_basic_shares', 'inc_weighted_average_diluted_shares', 'inc_basic_eps', 'inc_diluted_eps', 'inc_divi

In [14]:
print(raw.shape)
display(raw.describe(include='all').T)
raw.info(verbose=True, show_counts=True) # looked up the pandas documentation for this one
raw.nunique

# Missing value percentage per column
missing_pct = (raw.isna().mean()).sort_values(ascending=False)
display(missing_pct)
raw

(18274, 403)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
fiscal_year,18274.0,NaN,NaN,NaN,2023.045693,1.411906,2021.0,2022.0,2023.0,2024.0,2025.0
fiscal_period,18274,4,Q4,4611,NaN,NaN,NaN,NaN,NaN,NaN,NaN
period_end_date,18123,502,2023-09-30,741,NaN,NaN,NaN,NaN,NaN,NaN,NaN
inc_currency,18274,1,USD,18274,NaN,NaN,NaN,NaN,NaN,NaN,NaN
inc_revenue,18041.0,NaN,NaN,NaN,5043753590.254186,12741629281.782108,-3791000000.0,541131000.0,1554800000.0,4087000000.0,213386000000.0
...,...,...,...,...,...,...,...,...,...,...,...
cf_redevelopment_capital_expenditures,108.0,NaN,NaN,NaN,-43925157.814815,40296474.481358,-165607000.0,-71679000.0,-34371500.0,-11275000.0,0.0
buyback_quarter_api_flag,18274.0,NaN,NaN,NaN,0.403962,0.490703,0.0,0.0,0.0,1.0,1.0
buyback_quarter_share_reduction_flag,18274.0,NaN,NaN,NaN,0.12329,0.328779,0.0,0.0,0.0,0.0,1.0
buyback_quarter,18274.0,NaN,NaN,NaN,0.403962,0.490703,0.0,0.0,0.0,1.0,1.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18274 entries, 0 to 18273
Data columns (total 403 columns):
 #    Column                                                   Non-Null Count  Dtype  
---   ------                                                   --------------  -----  
 0    fiscal_year                                              18274 non-null  int64  
 1    fiscal_period                                            18274 non-null  object 
 2    period_end_date                                          18123 non-null  object 
 3    inc_currency                                             18274 non-null  object 
 4    inc_revenue                                              18041 non-null  float64
 5    inc_revenue_product                                      7507 non-null   float64
 6    inc_revenue_services                                     6700 non-null   float64
 7    inc_cost_of_revenue                                      14733 non-null  float64
 8    inc_gross_prof

inc_annuity_benefits_paid               1.0
bal_investment_properties_fair_value    1.0
inc_new_business_value                  1.0
website_url                             1.0
inc_reserve_life_years                  1.0
                                       ... 
cf_currency                             0.0
profile_api_ticker_used                 0.0
sic_code                                0.0
company_name                            0.0
fiscal_year                             0.0
Length: 403, dtype: float64

,fiscal_year,fiscal_period,period_end_date,inc_currency,inc_revenue,inc_revenue_product,inc_revenue_services,inc_cost_of_revenue,inc_gross_profit,inc_operating_expenses,inc_research_development,inc_selling_general_administrative,inc_selling_marketing_expenses,inc_general_admin_expenses,inc_depreciation_amortization,inc_depreciation,inc_amortization_of_intangibles,inc_stock_based_compensation,inc_operating_income,inc_interest_expense,inc_interest_income,inc_other_income_expense,inc_income_before_taxes,inc_income_tax_expense,inc_net_income_continuing_operations,inc_net_income_discontinued_operations,inc_net_income,inc_basic_shares_outstanding,inc_diluted_shares_outstanding,inc_weighted_average_basic_shares,inc_weighted_average_diluted_shares,inc_basic_eps,inc_diluted_eps,inc_dividends_per_share,inc_comprehensive_income,inc_other_comprehensive_income,inc_impairment_charges,inc_restructuring_costs,inc_net_income_noncontrolling_interest,inc_preferred_dividends,inc_net_income_available_to_common,inc_revenue_from_contracts_excluding_tax,inc_revenue_from_contracts_including_tax,inc_income_before_taxes_and_extraordinary,inc_other_comprehensive_income_net_of_tax,inc_comprehensive_income_net_of_tax,inc_interest_income_operating,ticker,api_ticker_used,bal_currency,bal_total_assets,bal_current_assets,bal_non_current_assets,bal_cash_and_equivalents,bal_short_term_investments,bal_long_term_investments,bal_accounts_receivable,bal_inventory,bal_property_plant_equipment,bal_goodwill,bal_intangible_assets,bal_other_current_assets,bal_other_non_current_assets,bal_prepaid_expenses,bal_total_liabilities,bal_current_liabilities,bal_non_current_liabilities,bal_accounts_payable,bal_accruals_current,bal_short_term_debt,bal_deferred_revenue_current,bal_other_current_liabilities,bal_long_term_debt,bal_deferred_tax_liabilities,bal_deferred_revenue_non_current,bal_other_non_current_liabilities,bal_accrued_liabilities,bal_total_equity,bal_retained_earnings,bal_additional_paid_in_capital,bal_treasury_stock,bal_treasury_shares,bal_accumulated_other_comprehensive_income,bal_noncontrolling_interest,bal_preferred_stock_value,bal_operating_lease_right_of_use_assets,bal_operating_lease_liability_current,bal_operating_lease_liability_noncurrent,bal_investment_securities_fair_value,bal_available_for_sale_debt_securities,bal_common_stock_shares_authorized,bal_shares_issued,bal_common_stock_par_value_per_share,bal_total_debt,bal_net_debt,bal_working_capital,bal_tangible_book_value,bal_book_value_per_share,cf_currency,cf_net_cash_from_operations,cf_deferred_taxes,cf_change_in_receivables,cf_change_in_inventory,cf_change_in_accounts_payable,cf_change_in_deferred_revenue,cf_other_operating_assets_change,cf_other_operating_liabilities_change,cf_tax_withholding_payments,cf_other_misc_operating,cf_net_cash_from_investing,cf_capital_expenditures,cf_acquisitions,cf_purchase_of_investments,cf_proceeds_from_investments,cf_recurring_capital_expenditures,cf_net_cash_from_financing,cf_issuance_of_debt,cf_repayment_of_debt,cf_repurchase_of_stock,cf_cash_at_beginning_of_period,cf_cash_at_end_of_period,cf_net_change_in_cash,cf_free_cash_flow,cf_levered_free_cash_flow,shr_basic_shares_outstanding,shr_diluted_shares_outstanding,shr_weighted_average_basic_shares,shr_weighted_average_diluted_shares,shr_treasury_shares,shr_dilution_percentage,shr_shares_change_vs_prior_period,shr_shares_change_percentage,shr_is_buyback,profile_api_ticker_used,sector,industry,sic_code,company_name,fiscal_year_end,website_url,exchange,inc_net_interest_income,inc_provision_for_credit_losses,inc_net_interest_margin,inc_return_on_average_assets,inc_return_on_average_equity,inc_fee_income,inc_trading_revenue,inc_interest_income_loans_commercial,inc_interest_income_loans_consumer,inc_interest_income_loans_total,inc_interest_income_securities,inc_interest_expense_deposits,inc_interest_expense_borrowings,inc_interest_expense_long_term_debt,inc_net_interest_income_after_provision,inc_investment_banking_revenue,inc_

In [15]:
# Let's start data cleaning and sorting through these 402 x variables....
df = lab.copy()

# Move the ticker column as the second column for better readability
column_to_move = df.pop("ticker")
df.insert(0, "Ticker", column_to_move)

# # Drop columns that add no material value (automatically)
# df = df.loc[:, df.nunique(dropna=False) > 1]

# # Drop columns that add no material value (manually)
# df = df.drop(columns=[
# ])

# # Drop all columns with more than >70% missing observations
# threshold = 0.7
# missing_pct = df.isna().mean()
# columns_to_drop = missing_pct[missing_pct > threshold].index
# df = df.drop(columns=columns_to_drop)

df.columns.tolist()

['Ticker',
 'fiscal_year',
 'fiscal_period',
 'period_end_date',
 'inc_currency',
 'inc_revenue',
 'inc_revenue_product',
 'inc_revenue_services',
 'inc_cost_of_revenue',
 'inc_gross_profit',
 'inc_operating_expenses',
 'inc_research_development',
 'inc_selling_general_administrative',
 'inc_selling_marketing_expenses',
 'inc_general_admin_expenses',
 'inc_depreciation_amortization',
 'inc_depreciation',
 'inc_amortization_of_intangibles',
 'inc_stock_based_compensation',
 'inc_operating_income',
 'inc_interest_expense',
 'inc_interest_income',
 'inc_other_income_expense',
 'inc_income_before_taxes',
 'inc_income_tax_expense',
 'inc_net_income_continuing_operations',
 'inc_net_income_discontinued_operations',
 'inc_net_income',
 'inc_basic_shares_outstanding',
 'inc_diluted_shares_outstanding',
 'inc_weighted_average_basic_shares',
 'inc_weighted_average_diluted_shares',
 'inc_basic_eps',
 'inc_diluted_eps',
 'inc_dividends_per_share',
 'inc_comprehensive_income',
 'inc_other_comprehen